# Save shallow-learning regression models + scaler (.pkl)

Trains the three regressors from `04_SL_regression.ipynb` (Linear Regression, XGBoost, Random Forest) on the Barcelona regression dataset and saves each one, together with the `StandardScaler` and the feature column order, as pickle files in `models/` (suffix `_regre` to keep them apart from the classification artifacts).

**Target: `noise_day` in dB** (continuous value, not the 0–4 class). Saving the scaler alongside the models is essential: without it, new input data (e.g. Milan / Viladecans / Berlin / Lyon) cannot be transformed into the space the models were trained on.

# Data and libraries

In [1]:
# standard packaging

import pandas as pd
import numpy as np
import pickle
import os

In [2]:
# load the dataset - original source

data = pd.read_csv("data/bcn_noise_regre_ml_dataset.csv")

In [3]:
data_clean = data.dropna(how = "any")
data_clean.reset_index(inplace=True, drop = True)
data_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 12854 entries, 0 to 12853
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   road_id                12854 non-null  str    
 1   noise_day              12854 non-null  int64  
 2   noise_evening          12854 non-null  int64  
 3   noise_night            12854 non-null  int64  
 4   road_category          12854 non-null  int64  
 5   dist_to_trunk          12854 non-null  float64
 6   dist_to_primary        12854 non-null  float64
 7   dist_to_secondary      12854 non-null  float64
 8   dist_to_tertiary       12854 non-null  float64
 9   dist_to_residential    12854 non-null  float64
 10  dist_to_living_street  12854 non-null  float64
 11  signals                12854 non-null  float64
 12  transport              12854 non-null  float64
 13  pois                   12854 non-null  float64
 14  width                  12854 non-null  float64
 15  betweenness  

### scaling

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [5]:
X = data_clean.drop(columns=['road_id','noise_day','noise_evening', 'noise_night'])
X_scaled = scaler.fit_transform(X)
X.head()

,road_category,dist_to_trunk,dist_to_primary,dist_to_secondary,dist_to_tertiary,dist_to_residential,dist_to_living_street,signals,transport,pois,width,betweenness,closeness_global,closeness_400,straightness,green,industrial,commercial
0,5,213.093233,509.628452,1110.052349,188.355179,0.0,385.612798,0.053298,0.000000,0.106597,50.000000,0.000039,0.000187,0.000017,0.807756,3.349277,0.0,0.0
1,5,173.078014,1621.681034,2110.530434,351.777378,0.0,117.669324,0.000000,0.014274,0.000000,20.006115,0.000224,0.000109,0.000004,0.741368,0.000000,0.0,0.0
2,3,904.118521,947.154817,0.000000,167.026629,0.0,19.810222,0.167307,0.000000,0.295484,50.000000,0.010452,0.000182,0.000018,0.831327,8.619238,0.0,0.0
3,5,1877.603526,414.648395,661.587375,10.597441,0.0,41.365655,0.000000,0.012453,0.002491,16.759462,0.004427,0.000215,0.000009,0.757172,4.346695,0.0,0.0
4,5,1872.177506,607.144241,967.746460,0.000000,0.0,49.109562,0.000000,0.053513,0.000000,50.000000,0.003735,0.000208,0.000006,0.736805,25.169641,0.0,0.0


In [6]:
y = np.asarray(data_clean["noise_day"])
y

array([55, 50, 65, ..., 70, 55, 55], shape=(12854,))

### splitting

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 42)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)

(10283, 18)
(10283,)
(2571, 18)


# Save scaler and feature column order

In [8]:
# output folder for all pickled artifacts
os.makedirs('models', exist_ok=True)

In [9]:
#SAVE SCALER FOR LATER USE

with open('models/Sscaler_regre.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [10]:
#save the feature column order - needed to arrange a new city's columns before scaler.transform

feature_columns = list(X.columns)
with open('models/feature_columns_regre.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

feature_columns

['road_category',
 'dist_to_trunk',
 'dist_to_primary',
 'dist_to_secondary',
 'dist_to_tertiary',
 'dist_to_residential',
 'dist_to_living_street',
 'signals',
 'transport',
 'pois',
 'width',
 'betweenness',
 'closeness_global',
 'closeness_400',
 'straightness',
 'green',
 'industrial',
 'commercial']

# Train and save models

In [11]:
# helper: print R2 / MAE / RMSE for a fitted regressor

from sklearn.metrics import mean_absolute_error, mean_squared_error

def regression_report(name, model):
    pred_test = model.predict(X_test)
    print(name)
    print(f"  R2 train: {model.score(X_train, y_train):.4f}")
    print(f"  R2 test:  {model.score(X_test, y_test):.4f}")
    print(f"  MAE test (dB):  {mean_absolute_error(y_test, pred_test):.3f}")
    print(f"  RMSE test (dB): {np.sqrt(mean_squared_error(y_test, pred_test)):.3f}")

## Linear Regression

In [12]:
from sklearn.linear_model import LinearRegression
linreg_model = LinearRegression()

linreg_model.fit(X_train, y_train)
regression_report('Linear Regression', linreg_model)

Linear Regression
  R2 train: 0.4479
  R2 test:  0.4610
  MAE test (dB):  4.099
  RMSE test (dB): 5.191


In [13]:
#save sk learn model

with open('models/linreg_regre_model.pkl', 'wb') as f:
    pickle.dump(linreg_model, f)

## XG boost

In [14]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    )

xgb_model.fit(X_train, y_train)
regression_report('XGBoost', xgb_model)

XGBoost
  R2 train: 0.8991
  R2 test:  0.6806
  MAE test (dB):  3.064
  RMSE test (dB): 3.996


In [15]:
#save xgboost model

with open('models/xgb_regre_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

## Random Forest

In [16]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    random_state=42,
    max_depth=None,
    min_samples_leaf=1,
    min_samples_split=2,
)

rf_model.fit(X_train, y_train)
regression_report('Random Forest', rf_model)

Random Forest
  R2 train: 0.9581
  R2 test:  0.7038
  MAE test (dB):  2.900
  RMSE test (dB): 3.848


In [17]:
#save sk learn model

with open('models/rf_regre_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Verify the saved files

Reload every pickle and check the loaded models reproduce the in-memory test R².

In [18]:
with open('models/Sscaler_regre.pkl', 'rb') as f:
    loaded_scaler = pickle.load(f)
with open('models/feature_columns_regre.pkl', 'rb') as f:
    loaded_columns = pickle.load(f)

# transform the raw (unscaled) features with the loaded scaler
X_check = loaded_scaler.transform(data_clean[loaded_columns])
_, X_check_test, _, y_check_test = train_test_split(X_check, y, test_size = 0.2, random_state = 42)

for name, path, trained in [
    ('Linear Regression', 'models/linreg_regre_model.pkl', linreg_model),
    ('XGBoost',           'models/xgb_regre_model.pkl',    xgb_model),
    ('Random Forest',     'models/rf_regre_model.pkl',     rf_model),
]:
    with open(path, 'rb') as f:
        loaded_model = pickle.load(f)
    r2_loaded = loaded_model.score(X_check_test, y_check_test)
    r2_memory = trained.score(X_test, y_test)
    assert abs(r2_loaded - r2_memory) < 1e-9, name
    print(f"{name}: loaded test R2 = {r2_loaded:.4f}  (matches in-memory model)")

Linear Regression: loaded test R2 = 0.4610  (matches in-memory model)


XGBoost: loaded test R2 = 0.6806  (matches in-memory model)
Random Forest: loaded test R2 = 0.7038  (matches in-memory model)


In [19]:
# demo: predict noise_day dB for 5 rows, as a deployment would

demo_raw = data_clean[loaded_columns].sample(5, random_state=0)
demo_scaled = loaded_scaler.transform(demo_raw)

with open('models/linreg_regre_model.pkl', 'rb') as f:
    linreg_loaded = pickle.load(f)

pd.DataFrame({
    'road_id': data_clean.loc[demo_raw.index, 'road_id'].values,
    'noise_day_true': data_clean.loc[demo_raw.index, 'noise_day'].values,
    'noise_day_pred': linreg_loaded.predict(demo_scaled).round(1),
})

,road_id,noise_day_true,noise_day_pred
0,30295888_30295890_0,50,57.4
1,30554956_30554852_0,55,60.0
2,1303465752_1303465757_0,60,60.5
3,1378060299_1378060258_0,60,61.0
4,1350466016_1350466211_0,45,44.4
